# NUV LED for FURST Flat Field
This notebook uses optika to estimate the ideal LED wavelength to use to generate a flat field image for FURST by matching the penetration depth of the NUV photon to that of a photon in the FURST wavelength range.
Two possibilities are considered:

1. No oxide on the silicon substate

2. A thin oxide layer upon the substrate, whose thickness is estimated using QE estimates (see Roy's paper)

Install optika

In [ ]:
!pip install optika

Import statements

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import astropy.units as u
import astropy.visualization
import named_arrays as na
import optika

FURST wavelength range

In [ ]:
wavelength = na.linspace(116.45, 183.89 , "wavelength", num=101) * u.nm

NUV LED wavelengths

In [ ]:
wavelength_led = na.linspace(300, 320, "led", num=101) * u.nm

Optical properties of Si

In [ ]:
si = optika.chemicals.Chemical("Si")

## No oxide layer

Compute penetration depths
$z = \frac{1}{\alpha}$

In [ ]:
depth = 1 / si.absorption(wavelength)
depth_led = 1 / si.absorption(wavelength_led)

Plot the resulting penetration depths for the two wavelength ranges

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax2 = ax.twiny()
    na.plt.plot(wavelength, depth, label="FURST", ax=ax)
    na.plt.plot(wavelength_led, depth_led, label='LED', ax=ax2, color='tab:orange')
    ax.set_xlabel(f"FURST wavelength ({ax.get_xlabel()})")
    ax2.set_xlabel(f"LED wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"penetration depth ({ax.get_ylabel()})")
    # ax.set_yscale("log")
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, loc=2)

Calculate the RMS difference of the penetration depths

In [ ]:
rms_difference = (depth - depth_led).rms("wavelength")

Plot the RMS difference

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(wavelength_led, rms_difference, ax=ax)
    ax.set_xlabel(f"LED wavelength ({ax.get_xlabel()})")
    ax.set_ylabel(f"RMS difference ({ax.get_ylabel()})")

Print the NUV LED wavelength that best matches the FURST penetration depth

In [ ]:
wavelength_led[np.argmin(rms_difference)].ndarray

## With oxide layer

Adjust NUV LED range

In [ ]:
wavelength_led = na.linspace(270, 320, "led", num=101) * u.nm

Load properties of the FURST CCD

In [ ]:
ccd = optika.sensors.materials.e2v_ccd97()

Estimated thickness of the oxide layer

In [ ]:
ccd.thickness_oxide

Define a function that computes the transmission coefficient for this oxide layer on the silicon substrate

In [ ]:
def transmission(wavelength):
  r, t = optika.materials.multilayer_efficiency(
      wavelength = wavelength,
      direction = 1,
      n = 1,
      layers = [optika.materials.Layer(chemical=ccd._chemical_oxide, thickness=ccd.thickness_oxide)],
      substrate = optika.materials.Layer(chemical=si, interface=optika.materials.profiles.ErfInterfaceProfile(ccd.roughness_substrate))
  )
  return t.average

Compute the transmission coefficient for the FURST wavelength range and the NUV LED wavelength range

In [ ]:
transmission_furst = transmission(wavelength)
transmission_led = transmission(wavelength_led)

Plot transmission coefficients for the two ranges, and the $1/e$ threshold (since that defines the penetration depth)

In [ ]:
with astropy.visualization.quantity_support():
  fig, ax = plt.subplots()
  ax2 = ax.twiny()
  na.plt.plot(wavelength, transmission_furst, ax=ax, label='FURST')
  na.plt.plot(wavelength_led, transmission_led, ax=ax2, color='tab:orange', label='LED')
  na.plt.axhline(1/np.e, color='dimgrey', ls='--', label='$1/e$')
  ax.set_xlabel(f"FURST wavelength ({ax.get_xlabel()})")
  ax2.set_xlabel(f"LED wavelength ({ax2.get_xlabel()})")
  ax.set_ylabel("transmission coefficient")
  h1, l1 = ax.get_legend_handles_labels()
  h2, l2 = ax2.get_legend_handles_labels()
  ax.legend(h1+h2, l1+l2, loc=2)

Calculate penetration depth in the case of an oxide layer for the two sets of wavelengths

Starting from Beer-Lambert Law and the definition of the penetration depth:

$$\frac{1}{e} =  T e^{-\alpha z}$$

where $T$ is the transmission coefficient, $\alpha$ is the absorption coefficient, and $z$ is the penetration depth.

$$\Rightarrow z = \frac{1 + \ln(T)}{\alpha}$$


In [ ]:
depth = (1 + np.log(transmission_furst)) / si.absorption(wavelength)
depth_led = (1 + np.log(transmission_led)) / si.absorption(wavelength_led)

Plot the resulting penetration depths for the two wavelength ranges

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    ax2 = ax.twiny()
    na.plt.plot(wavelength, depth, label="FURST", ax=ax)
    na.plt.plot(wavelength_led, depth_led, label='LED', ax=ax2, color='tab:orange')
    ax.set_xlabel(f"FURST wavelength ({ax.get_xlabel()})")
    ax2.set_xlabel(f"LED wavelength ({ax2.get_xlabel()})")
    ax.set_ylabel(f"penetration depth ({ax.get_ylabel()})")
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1+h2, l1+l2, loc=2)

Calculate the RMS difference of the penetration depths

In [ ]:
rms_difference = (depth - depth_led).rms("wavelength")

Plot the RMS difference

In [ ]:
with astropy.visualization.quantity_support():
    fig, ax = plt.subplots(constrained_layout=True)
    na.plt.plot(wavelength_led, (depth - depth_led).rms("wavelength"), ax=ax)
    ax.set_xlabel(f"LED wavelength ({ax.get_xlabel()})")
    ax.set_ylabel(f"RMS difference ({ax.get_ylabel()})")

Print the NUV LED wavelength that best matches the FURST penetration depth

In [ ]:
wavelength_led[np.argmin(rms_difference)].ndarray